In [6]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One Provider described as pure DATA."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS=[
    Provider("Groq", "GROQ_API_KEY", True, None, "openai/gpt-oss-20b"),
]


def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected=", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file")


def build_client(provider: Provider):
    from groq import Groq

    api_key=os.environ[provider.env_var]
    if provider.base_url is None:
        return Groq(api_key=api_key)
    return Groq(api_key=api_key, base_url=provider.base_url)


def llm_reply(prompt: str,*,max_tokens: int=300)->str:
    """Send one user prompt; return the assistant's text."""

    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model=provider.model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return result.choices[0].message.content

# Zero Shot Chain Of Thoughts

In [7]:
ZERO_SHOT_COT_SUFFIX="\n\nLet's think step by step"

def zero_shot_cot_prompt(question: str)->str:
    """This function takes a question and returns a prompt for zero-shot COT model."""
    return f"Question: {question}{ZERO_SHOT_COT_SUFFIX}"

In [8]:
QUESTION = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [9]:
print(zero_shot_cot_prompt(QUESTION))

Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?

Let's think step by step


In [10]:
reply=llm_reply(zero_shot_cot_prompt(QUESTION))
print(reply)

**Step 1 – Count the balls in the cans**

Roger buys **2** cans, and each can contains **3** tennis balls.

\[
2 \text{ cans} \times 3 \text{ balls/can} = 6 \text{ balls}
\]

**Step 2 – Add the new balls to the ones he already has**

He already had **5** tennis balls. Add the 6 new ones:

\[
5 \text{ (original)} + 6 \text{ (new)} = 11 \text{ balls}
\]

---

**Answer:** Roger now has **11 tennis balls**.


# Few Shot Chain Of Thoughts

In [11]:
FEW_SHOT_EXAMPLES="""
Q: There are 15 trees in the grove. Grover workers will plant 6 trees today. How many trees will be in the grove?
A: There are 15 trees to start. After planting 6 more, there are 15+6=21 trees.
Final Answer: 21

Q: Leah had 32 chocolates and her sister had 42. They ate 35. How many pieces do they have left in total?
A: Leah and her sister had 32+42=74 chocolates together. After eating 35, they have 74-35=39 left.
Final Answer: 39
""".strip()

In [15]:
TRICKY_QUESTION = (
    "A juggler can juggle 16 balls. Half of the balls are golf balls, and "
    "half of the golf balls are blue. How many blue golf balls are there?"
)

In [16]:
def few_shot_cot_prompt(question: str)->str:
    """This function takes a question and returns a prompt for few-shot COT model."""
    return f""" Solve each question by reasoning step by step.
    {FEW_SHOT_EXAMPLES}

    Question: {question}
    Answer:
    """

In [17]:
reply=llm_reply(few_shot_cot_prompt(TRICKY_QUESTION))
print(reply)

**Step 1: Determine how many golf balls there are.**  
- The juggler has a total of 16 balls.  
- Half of these are golf balls:  

\[
16 \times \frac{1}{2} = 8 \text{ golf balls}
\]

**Step 2: Find out how many of those golf balls are blue.**  
- Half of the golf balls are blue:  

\[
8 \times \frac{1}{2} = 4 \text{ blue golf balls}
\]

**Answer:**  
There are **4 blue golf balls**.
